# Cold Start Chat
## Chat system for addressing the cold start problem head on.
This will be using NLP to take in a user's input and parse any relevant information to then add to the users profile, e.g. they like horror films, they don't like Brad Pitt, etc.
This is to give them an immediate set of recommendations based on their profiles.

Another objective in these questions is to not ask redundant ones, e.g. if they've already said they don't like non-English films, we shouldn't ask them if they like French films.


### Libraries
- Recommendation library: LensKit
- NLP library: spaCy

In [6]:
import numpy as np
import lenskit
import pandas as pd
import spacy

print("NumPy version:", np.__version__)
print("LensKit version:", lenskit.__version__)
print("Pandas version:", pd.__version__)
print("spaCy version:", spacy.__version__)

NumPy version: 2.0.2
LensKit version: 0.14.4
Pandas version: 2.2.3
spaCy version: 3.8.2


In [7]:
nlp = spacy.load('en_core_web_sm')

In [8]:
# Initialize an empty user profile
user_profile = {
    'films_liked': [],
    'films_disliked': [],
    'genres_positive': [],
    'genres_negative': [],
    'actors_positive': [],
    'actors_negative': [],
    'directors_positive': [],
    'directors_negative': [],
    'languages_positive': [],
    'languages_negative': []
}

In [9]:
import spacy
from transformers import pipeline

# Load spaCy model
nlp = spacy.load('en_core_web_sm')

# Sentiment analysis pipeline
sentiment_analyzer = pipeline('sentiment-analysis')

def parse_user_input(user_input):
    """
    Parse user input for relevant information about likes and dislikes.
    """
    doc = nlp(user_input)
    entities = {'films': [], 'genres': [], 'actors': [], 'directors': [], 'languages': []}
    
    # Extract entities using spaCy's NER
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['actors'].append(ent.text)
        elif ent.label_ == 'WORK_OF_ART':
            entities['films'].append((ent.text, 'like'))  # Default to 'like', refine with sentiment analysis
        elif ent.label_ == 'LANGUAGE':
            entities['languages'].append(ent.text)
        # Add more entity types as needed

    # Analyze sentiment all entities
    for key in entities.keys():
        for i, entity in enumerate(entities[key]):
            sentiment = sentiment_analyzer(entity[0])[0]
            if sentiment['label'] == 'NEGATIVE':
                entities[key][i] = (entity, 'dislike')
            elif sentiment['label'] == 'POSITIVE':
                entities[key][i] = (entity, 'like')

    return entities

/Users/Cathal/Rec-Genie/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f (https://huggingface.co/distilbert/distilbert-base-uncased-finetuned-sst-2-english).
Using a pipeline without specifying a model name and revision in production is not recommended.
All PyTorch model weights were used when initializing TFDistilBertForSequenceClassification.

All the weights of TFDistilBertForSequenceClassification were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertForSequenceClassification for predictions without further training.


In [12]:
def update_user_profile(profile, parsed_data):
    """
    Update the user profile with parsed data.
    """
    # Add films to liked/disliked lists
    for film, preference in parsed_data['films']:
        if preference == 'like' and film not in profile['films_liked']:
            profile['films_liked'].append(film)
        elif preference == 'dislike' and film not in profile['films_disliked']:
            profile['films_disliked'].append(film)
    
    # Update other fields with unique entries on negative and positive lists
    for key in ['genres', 'actors', 'directors', 'languages']:
        for item in parsed_data[key]:
            if item not in profile[key + '_positive'] and item not in profile[key + '_negative']:
                profile[key + '_positive'].append(item)
                

    
        
    

    return profile


 

Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [('Sandra Bullock', 'like'), ('Brad Pitt', 'like')], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [('English', 'like')], 'languages_negative': []}


In [13]:
# Sample user input
user_input = "I love horror films and Sandra Bullock, but I can't stand Brad Pitt. I prefer movies in English, I don't like Chinese films."

# Parse user input
parsed_data = parse_user_input(user_input)

# Update the user profile
user_profile = update_user_profile(user_profile, parsed_data)

print("Updated User Profile:")
print(user_profile)


Updated User Profile:
{'films_liked': [], 'films_disliked': [], 'genres_positive': [], 'genres_negative': [], 'actors_positive': [('Sandra Bullock', 'like'), ('Brad Pitt', 'like')], 'actors_negative': [], 'directors_positive': [], 'directors_negative': [], 'languages_positive': [('English', 'like')], 'languages_negative': []}
